In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import pickle

In [48]:
df = pd.read_csv("Churn_Modelling.csv")

In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df.drop(['RowNumber',"CustomerId",'Surname'], axis=1, inplace=True)

In [6]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
df.Geography.value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [49]:
labelencoder_gender=LabelEncoder()
df['Gender']=labelencoder_gender.fit_transform(df['Gender'])
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [16]:
onehotencoder_geography=OneHotEncoder()
geo_encoded = onehotencoder_geography.fit_transform(df[['Geography']]).toarray()

In [18]:
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehotencoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [19]:
df = pd.concat([df.drop('Geography', axis=1),geo_encoded_df], axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [50]:
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(labelencoder_gender, file=file)

In [22]:
with open("onehot_encoder_geography.pkl","wb") as file:
    pickle.dump(obj=onehotencoder_geography, file=file)

In [23]:
x=df.drop(['Exited'],axis=1)
y=df['Exited']

In [24]:
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.2, random_state=42)

#scaling:
scaler=StandardScaler()
scaler.fit_transform(x_train)
scaler.transform(x_test)

#creating scaler pickle file
with open("scaler.pkl","wb") as file:
    pickle.dump(obj=scaler, file=file)

## ANN Implementation:

In [25]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [26]:
model = Sequential([
    Dense(64, activation='relu', input_shape=((x_train.shape[1]),)), #Hidden layer 1 connected to inputs
    Dense(32,activation='relu'),        #Hidden layer 2
    Dense(1,activation='sigmoid')       #output layer
    
    
])


d:\Projects\Churn Prediction\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [29]:
model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

In [47]:
#set up the tensorboard
log_dir="log/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [32]:
#set up early stopping
earlystopping_callback=EarlyStopping(patience=10,monitor='val_loss',restore_best_weights=True)

In [33]:
#model training:
history=model.fit(
    x_train,y_train,validation_data=(x_test,y_test),
    epochs=100, callbacks=[tensorflow_callback,earlystopping_callback]
)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.6845 - loss: 1100.4233 - val_accuracy: 0.7943 - val_loss: 214.3391
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6860 - loss: 168.0677 - val_accuracy: 0.5651 - val_loss: 103.6541
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6665 - loss: 170.3549 - val_accuracy: 0.7965 - val_loss: 70.7444
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6950 - loss: 70.3216 - val_accuracy: 0.4459 - val_loss: 44.2641
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6855 - loss: 54.8960 - val_accuracy: 0.7954 - val_loss: 48.0810
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6985 - loss: 62.2352 - val_accuracy: 0.4880 - val_loss: 25.0856
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6695 - loss: 23.4688 - val_accuracy: 0.7966 - val_loss: 55.9635
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6870 - loss: 28.8059 - va

In [34]:
model.save('model.h5')

In [44]:
%reload_ext tensorboard

In [46]:
%tensorboard --logdir log/fit

Reusing TensorBoard on port 6007 (pid 30540), started 0:07:30 ago. (Use '!kill 30540' to kill it.)